# 03 — Baselines (Semana 3) — pimple

## Objetivo
Validar o pipeline ponta a ponta (splits → load de imagens → target → baseline) e gerar métricas mínimas com rastreabilidade.

## Entradas (contrato do Notebook 02)
- `data/processed/train.csv`, `val.csv`, `test.csv`
- `data/processed/label_map.json`
- `data/processed/target_config.json`
- `data/processed/splits_report.json`

## Entregáveis (Notebook 03)
- `data/processed/baseline_metrics.json`
- `reports/baseline_summary.md`
- `reports/baseline_examples.png`


In [1]:
# 03_baselines.ipynb — Célula 02 (FIX: imports completos + helpers)

from __future__ import annotations

import json
import os
import sys
import time
import random
import platform
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

from PIL import Image, ImageOps
import matplotlib.pyplot as plt

# sklearn (métricas + baseline)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# progresso (opcional)
try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


SEED = 42
IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def now_utc_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def json_dump(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, sort_keys=True)


def json_load(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def ensure_exists(path: Path, kind: str = "path") -> None:
    if not path.exists():
        raise FileNotFoundError(f"[ERRO] {kind} não encontrado: {path}")


def resolve_image_path(images_dir: Path, project_root: Path, value: Any) -> Path:
    """
    Resolver ÚNICO para imagens.

    Suporta:
      1) path absoluto existente
      2) path relativo ao PROJECT_ROOT (ex.: data\\raw\\...)
      3) nome de arquivo dentro de images_dir
      4) stem (tenta extensões conhecidas)
      5) path absoluto que não existe aqui -> tenta pelo nome/stem em images_dir

    Retorna Path("__MISSING__") se falhar.
    """
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return Path("__MISSING__")

    s = str(value).strip().strip('"').strip("'")
    if not s:
        return Path("__MISSING__")

    p = Path(s)

    # 1) absoluto existe
    if p.is_absolute() and p.is_file():
        return p

    # 2) relativo ao PROJECT_ROOT
    if not p.is_absolute():
        cand = (project_root / p).resolve()
        if cand.is_file():
            return cand

    # 3) apenas filename
    cand2 = images_dir / p.name
    if cand2.is_file():
        return cand2

    # 4) tenta pelo stem dentro do images_dir
    stem = p.stem
    for ext in IMAGE_EXTS:
        cand3 = images_dir / f"{stem}{ext}"
        if cand3.is_file():
            return cand3

    return Path("__MISSING__")


set_global_seed(SEED)
print("OK — seed setada:", SEED)
print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
print("sklearn: imports OK")


OK — seed setada: 42
Python: 3.11.9 | Platform: Windows-10-10.0.26100-SP0
sklearn: imports OK


In [2]:
# 03_baselines.ipynb — Célula 03 (FIX robusto)

def _looks_like_pimple_root(p: Path) -> bool:
    return (
        (p / "data" / "raw" / "lesions" / "images").exists()
        and (p / "data" / "processed").exists()
    )


def find_project_root_robust() -> Path:
    env_root = os.environ.get("PIMPLE_PROJECT_ROOT") or os.environ.get("PROJECT_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if _looks_like_pimple_root(p):
            return p
        raise FileNotFoundError(
            f"[ERRO] Env PROJECT_ROOT/PIMPLE_PROJECT_ROOT aponta para {p}, "
            "mas não parece ser a raiz do repo (faltam data/raw/lesions/images ou data/processed)."
        )

    start = Path.cwd().resolve()
    chain = [start, *start.parents]

    for base in chain:
        if _looks_like_pimple_root(base):
            return base

    for base in chain:
        cand = base / "pimple"
        if _looks_like_pimple_root(cand):
            return cand

    tried = []
    for base in chain[:6]:
        tried.append(str(base))
        tried.append(str(base / "pimple"))

    raise FileNotFoundError(
        "Não consegui localizar PROJECT_ROOT.\n"
        f"- cwd: {start}\n"
        "- Procurei por marcadores: data/raw/lesions/images e data/processed\n"
        "- Caminhos tentados (amostra):\n  - " + "\n  - ".join(tried) + "\n\n"
        "Correção rápida: defina uma env antes de rodar:\n"
        "  os.environ['PIMPLE_PROJECT_ROOT'] = '/caminho/para/pimple'\n"
        "ou rode o Jupyter a partir de dentro do repo."
    )


PROJECT_ROOT = find_project_root_robust()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "lesions"
IMAGES_DIR = RAW_DIR / "images"
MASKS_DIR = RAW_DIR / "masks"  # não usado no baseline de classificação
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"  # fora do git

# garantir pastas de output ANTES do ensure
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Entradas do Notebook 02 (contrato)
TRAIN_CSV = PROCESSED_DIR / "train.csv"
VAL_CSV = PROCESSED_DIR / "val.csv"
TEST_CSV = PROCESSED_DIR / "test.csv"
LABEL_MAP_JSON = PROCESSED_DIR / "label_map.json"
TARGET_CONFIG_JSON = PROCESSED_DIR / "target_config.json"
SPLITS_REPORT_JSON = PROCESSED_DIR / "splits_report.json"

# Saídas obrigatórias (Notebook 03)
BASELINE_METRICS_JSON = PROCESSED_DIR / "baseline_metrics.json"
BASELINE_SUMMARY_MD = REPORTS_DIR / "baseline_summary.md"
BASELINE_EXAMPLES_PNG = REPORTS_DIR / "baseline_examples.png"

# Saída extra útil (não obrigatória)
BASELINE_CM_PNG = REPORTS_DIR / "baseline_confusion_matrix.png"

# Checagens de existência: inputs obrigatórios
for p, k in [
    (PROJECT_ROOT, "PROJECT_ROOT"),
    (RAW_DIR, "RAW_DIR"),
    (IMAGES_DIR, "IMAGES_DIR"),
    (PROCESSED_DIR, "PROCESSED_DIR"),
    (TRAIN_CSV, "train.csv"),
    (VAL_CSV, "val.csv"),
    (TEST_CSV, "test.csv"),
    (LABEL_MAP_JSON, "label_map.json"),
    (TARGET_CONFIG_JSON, "target_config.json"),
    (SPLITS_REPORT_JSON, "splits_report.json"),
]:
    ensure_exists(p, k)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("IMAGES_DIR:", IMAGES_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORTS_DIR:", REPORTS_DIR)
print("MODELS_DIR:", MODELS_DIR)


PROJECT_ROOT: C:\Users\win\Documents\GitHub\pimple
IMAGES_DIR: C:\Users\win\Documents\GitHub\pimple\data\raw\lesions\images
PROCESSED_DIR: C:\Users\win\Documents\GitHub\pimple\data\processed
REPORTS_DIR: C:\Users\win\Documents\GitHub\pimple\reports
MODELS_DIR: C:\Users\win\Documents\GitHub\pimple\models


In [3]:
# 03_baselines.ipynb — Célula 04 (FONTE ÚNICA: contrato + one-hot + policy + splits_report)

TARGET_CONTRACT_EFFECTIVE_JSON = REPORTS_DIR / "target_contract_effective.json"
TARGET_CONFIG_EFFECTIVE_JSON = PROCESSED_DIR / "target_config_effective.json"  # novo: não sobrescreve o original

target_config = json_load(TARGET_CONFIG_JSON)
label_map = json_load(LABEL_MAP_JSON)
splits_report = json_load(SPLITS_REPORT_JSON)

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Shapes:")
print("  train:", train_df.shape)
print("  val  :", val_df.shape)
print("  test :", test_df.shape)

# Política do Week-3 sanity:
ONE_HOT_POLICY = "abort"   # "abort" (recomendado) ou "filter"
ONE_HOT_ATOL = 1e-3
FIND_IMAGECOL_SAMPLE_N = 400


def pick(d: dict, *paths: str, default=None):
    for p in paths:
        cur = d
        ok = True
        for k in p.split("."):
            if isinstance(cur, dict) and k in cur:
                cur = cur[k]
            else:
                ok = False
                break
        if not ok:
            continue
        if cur is None:
            continue
        if isinstance(cur, str) and cur.strip() == "":
            continue
        if isinstance(cur, list) and len(cur) == 0:
            continue
        return cur
    return default


# --- contrato do target_config (suporta nesting em target_definition.*) ---
image_col_cfg = pick(target_config, "image_col", "columns.image_col", "target_definition.image_col")
label_cols = pick(target_config, "label_cols", "columns.label_cols", "target_definition.label_cols", default=[])
meta_cols = pick(target_config, "meta_cols", "columns.meta_cols", "target_definition.meta_cols", default=[])

mode = pick(target_config, "mode", "target_definition.mode")
target_encoding = pick(
    target_config,
    "target_encoding",
    "target_definition.target_encoding",
    "target_definition.source_encoding",
)

print("\nContrato target_config (resolved):")
print("  image_col (cfg):", image_col_cfg)
print("  n_label_cols:", len(label_cols))
print("  mode:", mode)
print("  target_encoding:", target_encoding)


def find_best_image_col(df: pd.DataFrame, sample_n: int, seed: int = SEED):
    """Inferência robusta: amostra aleatória (seed fixa), não head()."""
    if len(df) == 0:
        return "", 0.0, []
    n = min(sample_n, len(df))
    sample = df.sample(n=n, replace=False, random_state=seed)

    scores = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
        vals = sample[col].dropna()
        if vals.empty:
            continue

        ok = 0
        total = 0
        for v in vals.tolist():
            total += 1
            p = resolve_image_path(IMAGES_DIR, PROJECT_ROOT, v)  # <- inclui PROJECT_ROOT/value
            if str(p) != "__MISSING__":
                ok += 1

        ratio = ok / max(total, 1)
        if ratio > 0:
            scores.append((col, float(ratio)))

    scores.sort(key=lambda x: -x[1])
    if not scores:
        return "", 0.0, []
    return scores[0][0], scores[0][1], scores[:5]


def validate_schema_cols(split_name: str, df: pd.DataFrame, image_col: str, label_cols: list):
    missing = []
    if image_col not in df.columns:
        missing.append(image_col)
    for c in label_cols:
        if c not in df.columns:
            missing.append(c)
    if missing:
        raise KeyError(f"[ERRO] {split_name}: colunas ausentes: {sorted(set(missing))}")


def one_hot_integrity(split_name: str, df: pd.DataFrame, label_cols: list, atol: float):
    """Retorna payload e máscara ok (True=linha válida)."""
    y = df[label_cols].to_numpy(dtype=np.float32)
    row_sum = y.sum(axis=1)
    ok = np.isclose(row_sum, 1.0, atol=atol)
    n_bad = int((~ok).sum())

    min_val = float(np.nanmin(y)) if y.size else float("nan")
    max_val = float(np.nanmax(y)) if y.size else float("nan")

    payload = {
        "split": split_name,
        "n_rows": int(df.shape[0]),
        "n_bad_sum": n_bad,
        "bad_sum_ratio": float(n_bad / max(len(df), 1)),
        "min_label_val": min_val,
        "max_label_val": max_val,
        "atol": float(atol),
        "policy": ONE_HOT_POLICY,
    }

    # gate de faixa [0,1]
    if min_val < -1e-3 or max_val > 1.0 + 1e-3:
        raise ValueError(f"[GATE] {split_name}: valores fora de [0,1] (min={min_val}, max={max_val}).")

    return payload, ok


def apply_one_hot_policy(split_name: str, df: pd.DataFrame, ok_mask: np.ndarray, label_cols: list):
    n_bad = int((~ok_mask).sum())
    if n_bad == 0:
        return df

    bad_idx = np.where(~ok_mask)[0][:5]
    print(f"[ALERTA] {split_name}: {n_bad} linhas com soma(one-hot)!=1. Exibindo até 5:")
    display(df.iloc[bad_idx][label_cols].copy())

    if ONE_HOT_POLICY == "abort":
        raise ValueError(f"[GATE] one-hot inválido em {split_name}: n_bad_sum={n_bad}")
    if ONE_HOT_POLICY == "filter":
        df2 = df.loc[ok_mask].reset_index(drop=True)
        print(f"[FIX] {split_name}: removidas {n_bad} linhas inválidas. Novo shape={df2.shape}")
        return df2

    raise ValueError(f"[ERRO] ONE_HOT_POLICY inválida: {ONE_HOT_POLICY}")


# --- resolver image_col efetivo (sem “mágica” escondida: grava effective config) ---
if not isinstance(image_col_cfg, str) or image_col_cfg not in train_df.columns:
    best_col, best_ratio, top5 = find_best_image_col(train_df, sample_n=FIND_IMAGECOL_SAMPLE_N, seed=SEED)
    if not best_col or best_ratio < 0.80:
        raise ValueError(
            "[ERRO] image_col do config não existe no CSV e a inferência não foi confiável.\n"
            f"image_col_cfg={image_col_cfg}\n"
            f"Melhor candidato={best_col} (ratio={best_ratio:.2f})\n"
            f"Top-5={top5}\n"
            f"IMAGES_DIR={IMAGES_DIR}"
        )
    image_col = best_col
    print(f"\n[FIX] image_col do config ('{image_col_cfg}') não existe no train.csv.")
    print(f"[FIX] image_col efetivo recuperado do CSV: '{image_col}' (ratio_resolved={best_ratio:.2f})")
else:
    image_col = image_col_cfg

# valida schema em TODOS os splits
if not isinstance(label_cols, list) or len(label_cols) == 0:
    raise ValueError("[ERRO] target_config sem label_cols.")
validate_schema_cols("train", train_df, image_col, label_cols)
validate_schema_cols("val", val_df, image_col, label_cols)
validate_schema_cols("test", test_df, image_col, label_cols)

# integridade + policy (FONTE ÚNICA)
integrity_train, ok_train = one_hot_integrity("train", train_df, label_cols, atol=ONE_HOT_ATOL)
integrity_val, ok_val = one_hot_integrity("val", val_df, label_cols, atol=ONE_HOT_ATOL)
integrity_test, ok_test = one_hot_integrity("test", test_df, label_cols, atol=ONE_HOT_ATOL)

# aplica policy (abort/filter)
train_df = apply_one_hot_policy("train", train_df, ok_train, label_cols)
val_df = apply_one_hot_policy("val", val_df, ok_val, label_cols)
test_df = apply_one_hot_policy("test", test_df, ok_test, label_cols)

# atualiza rows após policy (para rastreabilidade)
integrity_train["n_rows_after_policy"] = int(train_df.shape[0])
integrity_val["n_rows_after_policy"] = int(val_df.shape[0])
integrity_test["n_rows_after_policy"] = int(test_df.shape[0])

# escreve target_contract_effective.json (com policy/atol) + info do splits_report
contract_effective = {
    "resolved_at_utc": now_utc_iso(),
    "source_target_config_json": str(TARGET_CONFIG_JSON),
    "image_col_from_config": image_col_cfg,
    "image_col_effective": image_col,
    "label_cols": label_cols,
    "meta_cols": meta_cols if isinstance(meta_cols, list) else [],
    "mode": mode,
    "target_encoding": target_encoding,
    "find_best_image_col": {
        "sample_n": FIND_IMAGECOL_SAMPLE_N,
        "strategy": "random_sample",
        "seed": SEED,
    },
    "one_hot_integrity": {
        "policy": ONE_HOT_POLICY,
        "atol": ONE_HOT_ATOL,
    },
    "splits_report_meta": {
        "keys": list(splits_report.keys()) if isinstance(splits_report, dict) else str(type(splits_report)),
        "seed": splits_report.get("seed") if isinstance(splits_report, dict) else None,
    },
}
json_dump(contract_effective, TARGET_CONTRACT_EFFECTIVE_JSON)

# (opcional, mas recomendado) grava um target_config_effective.json sem mexer no original do NB02
# Isso elimina o "mágico": os próximos notebooks podem usar o effective config.
target_config_effective = dict(target_config)
td = target_config_effective.get("target_definition") if isinstance(target_config_effective.get("target_definition"), dict) else None
if td is not None:
    td = dict(td)
    td["image_col"] = image_col
    target_config_effective["target_definition"] = td
else:
    target_config_effective["image_col"] = image_col

target_config_effective["effective_notes"] = {
    "generated_at_utc": now_utc_iso(),
    "derived_from": str(TARGET_CONFIG_JSON),
    "reason": "Align image_col to actual split CSV columns; avoid magic fallback in later notebooks.",
}
json_dump(target_config_effective, TARGET_CONFIG_EFFECTIVE_JSON)

print("\nContrato EFETIVO:")
print("  image_col:", image_col)
print("  n_label_cols:", len(label_cols))
print("  one_hot_policy:", ONE_HOT_POLICY, "| atol:", ONE_HOT_ATOL)
print("Salvo:", TARGET_CONTRACT_EFFECTIVE_JSON)
print("Salvo:", TARGET_CONFIG_EFFECTIVE_JSON)


Shapes:
  train: (7011, 12)
  val  : (1502, 12)
  test : (1502, 12)

Contrato target_config (resolved):
  image_col (cfg): image
  n_label_cols: 7
  mode: single_label
  target_encoding: one_hot_multiclass

[FIX] image_col do config ('image') não existe no train.csv.
[FIX] image_col efetivo recuperado do CSV: 'image_stem' (ratio_resolved=1.00)

Contrato EFETIVO:
  image_col: image_stem
  n_label_cols: 7
  one_hot_policy: abort | atol: 0.001
Salvo: C:\Users\win\Documents\GitHub\pimple\reports\target_contract_effective.json
Salvo: C:\Users\win\Documents\GitHub\pimple\data\processed\target_config_effective.json


## Gate — Integridade do target (single-label one-hot)

Validações mínimas:
- `label_cols` existem e são numéricas.
- Soma por linha em `label_cols` ≈ 1 (single-label).
- `image_col` está preenchida.


In [4]:
# 03_baselines.ipynb — Célula 05 (FIX DEFINITIVO: NÃO sobrescrever validate_one_hot / integrity_*)
# integrity_train/val/test é calculado UMA ÚNICA VEZ na célula de contrato/one-hot (Célula 04).
# Esta célula só deriva y_* e imprime um sanity de distribuição.

def infer_class_names(label_map_obj: Any, label_cols: List[str]) -> List[str]:
    if isinstance(label_map_obj, dict):
        if isinstance(label_map_obj.get("index_to_label"), list) and len(label_map_obj["index_to_label"]) > 0:
            return [str(x) for x in label_map_obj["index_to_label"]]
        if isinstance(label_map_obj.get("labels"), list) and len(label_map_obj["labels"]) > 0:
            return [str(x) for x in label_map_obj["labels"]]
    return [str(c) for c in label_cols]

class_names = infer_class_names(label_map, label_cols)
n_classes = len(label_cols)

def to_y_int(df: pd.DataFrame, label_cols: List[str]) -> np.ndarray:
    y_mat = df[label_cols].to_numpy(dtype=np.float32)
    return np.argmax(y_mat, axis=1).astype(np.int64)

y_train = to_y_int(train_df, label_cols)
y_val = to_y_int(val_df, label_cols)
y_test = to_y_int(test_df, label_cols)

print("Classes (index -> name):")
for i, name in enumerate(class_names):
    print(f"  {i}: {name}")

print("\nDistribuição (train):")
counts = pd.Series(y_train).value_counts().sort_index()
for i in range(n_classes):
    print(f"  {i:>2} {class_names[i]:<12} -> {int(counts.get(i, 0))}")

print("\nIntegridade (fonte única da célula de contrato/one-hot):")
print("  train:", integrity_train)
print("  val  :", integrity_val)
print("  test :", integrity_test)


Classes (index -> name):
  0: mel
  1: nv
  2: bcc
  3: akiec
  4: bkl
  5: df
  6: vasc

Distribuição (train):
   0 mel          -> 779
   1 nv           -> 4693
   2 bcc          -> 360
   3 akiec        -> 229
   4 bkl          -> 769
   5 df           -> 81
   6 vasc         -> 100

Integridade (fonte única da célula de contrato/one-hot):
  train: {'split': 'train', 'n_rows': 7011, 'n_bad_sum': 0, 'bad_sum_ratio': 0.0, 'min_label_val': 0.0, 'max_label_val': 1.0, 'atol': 0.001, 'policy': 'abort', 'n_rows_after_policy': 7011}
  val  : {'split': 'val', 'n_rows': 1502, 'n_bad_sum': 0, 'bad_sum_ratio': 0.0, 'min_label_val': 0.0, 'max_label_val': 1.0, 'atol': 0.001, 'policy': 'abort', 'n_rows_after_policy': 1502}
  test : {'split': 'test', 'n_rows': 1502, 'n_bad_sum': 0, 'bad_sum_ratio': 0.0, 'min_label_val': 0.0, 'max_label_val': 1.0, 'atol': 0.001, 'policy': 'abort', 'n_rows_after_policy': 1502}


In [5]:
# 03_baselines.ipynb — Célula 07 (PATCH: resolver único + sanity)

def add_image_paths(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    out = df.copy()
    out["_image_path"] = out[image_col].apply(lambda v: resolve_image_path(IMAGES_DIR, PROJECT_ROOT, v))

    missing_mask = out["_image_path"].astype(str) == "__MISSING__"
    missing = int(missing_mask.sum())

    if missing > 0:
        sample = out.loc[missing_mask, [image_col]].head(10)
        raise FileNotFoundError(
            f"[ERRO] {split_name}: {missing} imagens não resolvidas.\n"
            f"Exemplo (até 10):\n{sample}\n\n"
            f"IMAGES_DIR={IMAGES_DIR}\nPROJECT_ROOT={PROJECT_ROOT}"
        )
    return out


train_df2 = add_image_paths(train_df, "train")
val_df2 = add_image_paths(val_df, "val")
test_df2 = add_image_paths(test_df, "test")

# sanity: abrir algumas imagens reais
sample_n = 5
idxs = np.random.choice(len(train_df2), size=min(sample_n, len(train_df2)), replace=False)

for i in idxs:
    p = Path(train_df2.iloc[int(i)]["_image_path"])
    with Image.open(p) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        print("OK load:", p.name, "| size:", im.size, "| mode:", im.mode)


OK load: isic_0030088.jpg | size: (600, 450) | mode: RGB
OK load: isic_0029728.jpg | size: (600, 450) | mode: RGB
OK load: isic_0027348.jpg | size: (600, 450) | mode: RGB
OK load: isic_0027786.jpg | size: (600, 450) | mode: RGB
OK load: isic_0033177.jpg | size: (600, 450) | mode: RGB


## Baseline — Features simples + Logistic Regression

Features por imagem:
- Resize fixo (128×128) só para padronizar o cálculo
- Histograma por canal RGB (bins=16) com densidade
- Média e desvio padrão por canal

Modelo:
- `StandardScaler` + `LogisticRegression(solver="saga", class_weight="balanced")`


In [6]:
# 03_baselines.ipynb — Célula 09 (PATCH: histogram range 0..255 inclusive)

FEATURE_IMAGE_SIZE = (128, 128)
HIST_BINS = 16


def extract_features_from_path(path: Path) -> np.ndarray:
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im = im.resize(FEATURE_IMAGE_SIZE, resample=Image.BILINEAR)
        arr = np.asarray(im, dtype=np.uint8)

    feats = []

    # histograma por canal — range=(0,256) inclui 255
    for c in range(3):
        h, _ = np.histogram(arr[..., c], bins=HIST_BINS, range=(0, 256), density=True)
        feats.append(h.astype(np.float32))

    # mean/std (0..1)
    arr_f = (arr.astype(np.float32) / 255.0).reshape(-1, 3)
    mean = arr_f.mean(axis=0)
    std = arr_f.std(axis=0)
    feats.append(mean.astype(np.float32))
    feats.append(std.astype(np.float32))

    return np.concatenate(feats, axis=0).astype(np.float32)


def build_feature_matrix(df: pd.DataFrame, split_name: str) -> np.ndarray:
    paths = [Path(p) for p in df["_image_path"].tolist()]
    iterator = paths if tqdm is None else tqdm(paths, desc=f"Features ({split_name})", total=len(paths))

    X = np.zeros((len(paths), 3 * HIST_BINS + 6), dtype=np.float32)
    for i, p in enumerate(iterator):
        X[i] = extract_features_from_path(p)
    return X


X_train = build_feature_matrix(train_df2, "train")
X_val = build_feature_matrix(val_df2, "val")
X_test = build_feature_matrix(test_df2, "test")

print("X shapes:", X_train.shape, X_val.shape, X_test.shape)


Features (train):   0%|          | 0/7011 [00:00<?, ?it/s]

Features (val):   0%|          | 0/1502 [00:00<?, ?it/s]

Features (test):   0%|          | 0/1502 [00:00<?, ?it/s]

X shapes: (7011, 54) (1502, 54) (1502, 54)


In [7]:
# 03_baselines.ipynb — Célula 10

def eval_predictions(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro")),
    }


majority_class = int(pd.Series(y_train).value_counts().idxmax())
y_val_major = np.full_like(y_val, fill_value=majority_class)
y_test_major = np.full_like(y_test, fill_value=majority_class)

metrics_major_val = eval_predictions(y_val, y_val_major)
metrics_major_test = eval_predictions(y_test, y_test_major)

print("Majority baseline — val :", metrics_major_val)
print("Majority baseline — test:", metrics_major_test)

clf = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            solver="saga",
            penalty="l2",
            C=1.0,
            max_iter=3000,
            class_weight="balanced",
            n_jobs=-1,
            random_state=SEED,
        )),
    ]
)

t0 = time.time()
clf.fit(X_train, y_train)
train_time_s = time.time() - t0

y_val_pred = clf.predict(X_val)
y_test_pred = clf.predict(X_test)

metrics_lr_val = eval_predictions(y_val, y_val_pred)
metrics_lr_test = eval_predictions(y_test, y_test_pred)

print("\nLogReg baseline — train_time_s:", round(train_time_s, 2))
print("LogReg baseline — val :", metrics_lr_val)
print("LogReg baseline — test:", metrics_lr_test)

print("\nClassification report (test):")
print(classification_report(y_test, y_test_pred, target_names=class_names, digits=4))


Majority baseline — val : {'accuracy': 0.6697736351531292, 'f1_macro': 0.11460469355206197}
Majority baseline — test: {'accuracy': 0.6697736351531292, 'f1_macro': 0.11460469355206197}


c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)



LogReg baseline — train_time_s: 16.36
LogReg baseline — val : {'accuracy': 0.4826897470039947, 'f1_macro': 0.31961095855672156}
LogReg baseline — test: {'accuracy': 0.48202396804260983, 'f1_macro': 0.33904869510914043}

Classification report (test):
              precision    recall  f1-score   support

         mel     0.2981    0.5569    0.3883       167
          nv     0.9492    0.4831    0.6403      1006
         bcc     0.2262    0.4935    0.3102        77
       akiec     0.1152    0.3878    0.1776        49
         bkl     0.3333    0.4000    0.3636       165
          df     0.0732    0.3529    0.1212        17
        vasc     0.2462    0.7619    0.3721        21

    accuracy                         0.4820      1502
   macro avg     0.3202    0.4909    0.3390      1502
weighted avg     0.7251    0.4820    0.5403      1502



c:\Users\win\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [8]:
# 03_baselines.ipynb — Célula 11

cm = confusion_matrix(y_test, y_test_pred, labels=list(range(n_classes)))

fig = plt.figure(figsize=(9, 7))
ax = plt.gca()
ax.imshow(cm)

ax.set_title("Confusion Matrix — Test (LogReg baseline)")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")

ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticklabels(class_names)

for i in range(n_classes):
    for j in range(n_classes):
        ax.text(j, i, str(int(cm[i, j])), ha="center", va="center", fontsize=8)

plt.tight_layout()
fig.savefig(BASELINE_CM_PNG, dpi=160)
plt.close(fig)

print("Salvo:", BASELINE_CM_PNG)


Salvo: C:\Users\win\Documents\GitHub\pimple\reports\baseline_confusion_matrix.png


In [9]:
# 03_baselines.ipynb — Célula 12

EXAMPLES_N = 12
GRID_ROWS, GRID_COLS = 3, 4

rng = np.random.default_rng(SEED)
idxs = rng.choice(len(test_df2), size=min(EXAMPLES_N, len(test_df2)), replace=False)

proba = clf.predict_proba(X_test) if hasattr(clf, "predict_proba") else None

fig = plt.figure(figsize=(14, 10))

for k, idx in enumerate(idxs):
    row = test_df2.iloc[int(idx)]
    img_path = Path(row["_image_path"])
    true_i = int(y_test[int(idx)])
    pred_i = int(y_test_pred[int(idx)])

    with Image.open(img_path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im_disp = im.resize((224, 224), resample=Image.BILINEAR)

    ax = plt.subplot(GRID_ROWS, GRID_COLS, k + 1)
    ax.imshow(im_disp)
    ax.axis("off")

    conf = ""
    if proba is not None:
        conf_val = float(np.max(proba[int(idx)]))
        conf = f" | conf={conf_val:.2f}"

    ok = "✅" if pred_i == true_i else "❌"
    ax.set_title(
        f"{ok} true={class_names[true_i]} | pred={class_names[pred_i]}{conf}",
        fontsize=10
    )

plt.suptitle("Baseline Examples — Test (LogReg features)", fontsize=14)
plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(BASELINE_EXAMPLES_PNG, dpi=160)
plt.close(fig)

print("Salvo:", BASELINE_EXAMPLES_PNG)


C:\Users\win\AppData\Local\Temp\ipykernel_2288\3335559903.py:39: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
C:\Users\win\AppData\Local\Temp\ipykernel_2288\3335559903.py:39: UserWarning: Glyph 10060 (\N{CROSS MARK}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.96])
C:\Users\win\AppData\Local\Temp\ipykernel_2288\3335559903.py:40: UserWarning: Glyph 9989 (\N{WHITE HEAVY CHECK MARK}) missing from font(s) DejaVu Sans.
  fig.savefig(BASELINE_EXAMPLES_PNG, dpi=160)
C:\Users\win\AppData\Local\Temp\ipykernel_2288\3335559903.py:40: UserWarning: Glyph 10060 (\N{CROSS MARK}) missing from font(s) DejaVu Sans.
  fig.savefig(BASELINE_EXAMPLES_PNG, dpi=160)


Salvo: C:\Users\win\Documents\GitHub\pimple\reports\baseline_examples.png


In [10]:
# 03_baselines.ipynb — Célula 13 (FIX: hash do splits_report + reforço do effective config)

import hashlib

def file_stat(path: Path) -> Dict[str, Any]:
    st = path.stat()
    return {
        "path": str(path),
        "size_bytes": int(st.st_size),
        "mtime_utc": datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat(),
    }

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _ratio(payload: Dict[str, Any]) -> float:
    n_bad = float(payload.get("n_bad_sum", 0))
    n_rows = float(payload.get("n_rows_after_policy", payload.get("n_rows", 1)))
    return float(payload.get("bad_sum_ratio", n_bad / max(n_rows, 1.0)))

splits_meta = {
    "seed": splits_report.get("seed") if isinstance(splits_report, dict) else None,
    "keys": list(splits_report.keys()) if isinstance(splits_report, dict) else str(type(splits_report)),
    "splits_report_sha256": sha256_file(SPLITS_REPORT_JSON),
}

metrics_payload = {
    "created_at_utc": now_utc_iso(),
    "project_root": str(PROJECT_ROOT),
    "inputs": {
        "train_csv": file_stat(TRAIN_CSV),
        "val_csv": file_stat(VAL_CSV),
        "test_csv": file_stat(TEST_CSV),
        "label_map_json": file_stat(LABEL_MAP_JSON),
        "target_config_json": file_stat(TARGET_CONFIG_JSON),
        "target_config_effective_json": str(TARGET_CONFIG_EFFECTIVE_JSON),
        "splits_report_json": file_stat(SPLITS_REPORT_JSON),
        "target_contract_effective_json": str(TARGET_CONTRACT_EFFECTIVE_JSON),
    },
    "splits_report_meta": splits_meta,
    "target_contract": {
        "mode": mode,
        "target_encoding": target_encoding,
        "image_col": image_col,
        "label_cols": label_cols,
        "meta_cols": meta_cols if isinstance(meta_cols, list) else [],
        "n_classes": n_classes,
        "class_names": class_names,
    },
    "integrity": {
        "train": integrity_train,
        "val": integrity_val,
        "test": integrity_test,
    },
    "notes": {
        "target_config_effective_usage": (
            "Notebook 03 gera target_config_effective.json para eliminar 'mágica' de fallback. "
            "A partir do Notebook 04, usar esse arquivo como contrato preferencial."
        )
    },
    "baselines": {
        "majority_class": {
            "val": metrics_major_val,
            "test": metrics_major_test,
        },
        "logreg_features": {
            "train_time_s": float(train_time_s),
            "val": metrics_lr_val,
            "test": metrics_lr_test,
            "confusion_matrix_test": cm.tolist(),
        },
    },
    "artifacts": {
        "baseline_metrics_json": str(BASELINE_METRICS_JSON),
        "baseline_summary_md": str(BASELINE_SUMMARY_MD),
        "baseline_examples_png": str(BASELINE_EXAMPLES_PNG),
        "baseline_confusion_matrix_png": str(BASELINE_CM_PNG),
    },
    "env": {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
}

json_dump(metrics_payload, BASELINE_METRICS_JSON)
print("Salvo:", BASELINE_METRICS_JSON)

policy = str(integrity_train.get("policy", "unknown"))
atol = integrity_train.get("atol", "n/a")

tr_bad = int(integrity_train.get("n_bad_sum", 0))
va_bad = int(integrity_val.get("n_bad_sum", 0))
te_bad = int(integrity_test.get("n_bad_sum", 0))

tr_ratio = _ratio(integrity_train)
va_ratio = _ratio(integrity_val)
te_ratio = _ratio(integrity_test)

tr_rows = int(integrity_train.get("n_rows_after_policy", integrity_train.get("n_rows", 0)))
va_rows = int(integrity_val.get("n_rows_after_policy", integrity_val.get("n_rows", 0)))
te_rows = int(integrity_test.get("n_rows_after_policy", integrity_test.get("n_rows", 0)))

seed_in_splits = splits_meta.get("seed", "n/a")
hash_splits = splits_meta["splits_report_sha256"]

summary_md = f"""# Baseline Summary - pimple (Notebook 03)

Created (UTC): {metrics_payload["created_at_utc"]}
Seed (notebook): {SEED}
Seed (splits_report): {seed_in_splits}
splits_report_sha256: {hash_splits}

## Target contract (effective)
- mode: {mode}
- target_encoding: {target_encoding}
- image_col: {image_col}
- n_classes: {n_classes}
- classes: {", ".join(class_names)}

## Effective config (eliminate magic)
- target_config_effective.json: {TARGET_CONFIG_EFFECTIVE_JSON}
  - Recommendation: use this file from Notebook 04 onward.

## Target integrity (one-hot)
- policy: {policy} (atol={atol})
- train: rows={tr_rows} | n_bad_sum={tr_bad} | ratio={tr_ratio:.6f}
- val: rows={va_rows} | n_bad_sum={va_bad} | ratio={va_ratio:.6f}
- test: rows={te_rows} | n_bad_sum={te_bad} | ratio={te_ratio:.6f}

## Metrics (val / test)

| Baseline | Accuracy (val) | F1 macro (val) | Accuracy (test) | F1 macro (test) |
|---|---:|---:|---:|---:|
| Majority | {metrics_major_val["accuracy"]:.4f} | {metrics_major_val["f1_macro"]:.4f} | {metrics_major_test["accuracy"]:.4f} | {metrics_major_test["f1_macro"]:.4f} |
| LogReg (features) | {metrics_lr_val["accuracy"]:.4f} | {metrics_lr_val["f1_macro"]:.4f} | {metrics_lr_test["accuracy"]:.4f} | {metrics_lr_test["f1_macro"]:.4f} |

## Artifacts
- data/processed/baseline_metrics.json
- reports/baseline_summary.md
- reports/baseline_examples.png
- reports/baseline_confusion_matrix.png
"""

BASELINE_SUMMARY_MD.parent.mkdir(parents=True, exist_ok=True)
BASELINE_SUMMARY_MD.write_text(summary_md, encoding="utf-8")
print("Salvo:", BASELINE_SUMMARY_MD)


Salvo: C:\Users\win\Documents\GitHub\pimple\data\processed\baseline_metrics.json
Salvo: C:\Users\win\Documents\GitHub\pimple\reports\baseline_summary.md
